# Prepare Table 010

Download and clean this ABS source, save its workbook and CSV, then publish its Delta table directly. Each notebook runs independently.

Use the job parameters described in [README](../README.md), and fill in `catalog` and `schema` in the final publishing cell.

Table 010 covers the civilian population aged 15 years and over.


## 1. Source settings

Edit these values when ABS changes this source. Review the workbook before updating the reference file. Each source keeps its own layout and parsing code.

In [ ]:
# Set this source's name, reference filename and ABS download filename.
dataset = "Table 010"
dataset_key = "table_010"
baseline_file = "Table 010.xlsx"
download_file = "62020010.xlsx"
base_url = "https://www.abs.gov.au/statistics/labour/employment-and-unemployment/labour-force-australia/"

# List the exact data sheets and the metadata rows in workbook order.
data_sheets = ["Data1", "Data2", "Data3"]
metadata_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start", "series_end", "n_obs", "series_id"]

## 2. Shared job parameters

The job supplies `release_month` (YYYY-MM), `run_date` (`{{job.start_time.iso_date}}`, the job start date in UTC), `job_run_id` (`{{job.run_id}}`) and `workspace_path` (the absolute `/Workspace/...` path to the project folder). Keep the reference workbook in its `Reference Datasets/` folder; downloaded workbooks and cleaned CSVs are saved under `outputs/runs/`. Install pandas, openpyxl and requests in the task environment.

In [ ]:
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests
from IPython.display import display

# Read the shared settings supplied by the Databricks job.
dbutils.widgets.text("release_month", "", "ABS reference month (YYYY-MM)")
dbutils.widgets.text("run_date", "", "Job run date (YYYY-MM-DD, UTC)")
dbutils.widgets.text("job_run_id", "", "Job run ID")
dbutils.widgets.text("workspace_path", "", "Workspace project path")
release_month = dbutils.widgets.get("release_month")
run_date = dbutils.widgets.get("run_date")
job_run_id = dbutils.widgets.get("job_run_id")
workspace = Path(dbutils.widgets.get("workspace_path"))
download_month = pd.Timestamp(datetime.strptime(release_month, "%Y-%m"))
assert release_month == download_month.strftime("%Y-%m"), "Use YYYY-MM for release_month"
assert job_run_id and "{{" not in job_run_id, "Supply the resolved job run ID"
assert dbutils.widgets.get("workspace_path"), "Supply workspace_path"

# Set the reference folder and prefix the output folder with the job start date.
baseline = workspace / "Reference Datasets"
run = workspace / "outputs" / "runs" / f"{run_date}_{job_run_id}" / dataset_key
print(dataset, release_month, run)

## 3. Download the release workbook

In [ ]:
# Build the download address from this source's base URL, release month and filename.
source_url = base_url + f"{download_month:%b-%Y}/".lower() + download_file
response = requests.get(source_url, timeout=120, headers={"Cache-Control": "no-cache"})
response.raise_for_status()
assert response.content.startswith(b"PK"), f"{dataset}: download is not an XLSX file"

# Save the original workbook after the download succeeds.
(run / "raw").mkdir(parents=True, exist_ok=True)
(run / "raw" / download_file).write_bytes(response.content)
print(source_url)

## 4. Read, check and reshape the workbook

Keep the existing structural and series-definition checks. The release-month check prevents this source from being saved under a different month.

In [ ]:
# Read every sheet from the reference and downloaded workbooks.
table_010_base = pd.read_excel(baseline / baseline_file, sheet_name=None, header=None)
table_010_book = pd.read_excel(run / "raw" / download_file, sheet_name=None, header=None)

# Check that the downloaded workbook has the same sheet names as the reference.
dataset = "Table 010"
assert set(table_010_book) == set(table_010_base), f"{dataset}: workbook sheets changed"


# Locate the first monthly data row and the row containing series IDs.
first_data_row = len(metadata_fields)
series_id_row = metadata_fields.index("series_id")
# Select the series details to compare, excluding end dates and observation counts.
definition_fields = ["description", "unit", "series_type", "data_type", "frequency", "collection_month", "series_start"]

# Create empty lists to collect series details, monthly values and sheet summaries.
metadata_frames, observation_frames, sheet_checks = [], [], []

# Read the three data sheets by name.
for sheet in data_sheets:
    raw_sheet = table_010_book[sheet]
    # Check that the first ten row labels match those in the reference sheet.
    baseline_sheet = table_010_base[sheet]
    pd.testing.assert_series_equal(raw_sheet.iloc[:first_data_row, 0], baseline_sheet.iloc[:first_data_row, 0], obj=f"{dataset}/{sheet}: header rows")
    # Read the series IDs and check that none are missing or repeated within this sheet.
    series_ids = raw_sheet.iloc[series_id_row, 1:]
    assert series_ids.notna().all() and series_ids.is_unique, f"{dataset}/{sheet}: missing or duplicate series IDs"

    # Turn the series details into one row per series, label the columns and sort by ID.
    sheet_metadata = raw_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    baseline_metadata = baseline_sheet.iloc[:first_data_row, 1:].T.set_axis(metadata_fields, axis=1).set_index("series_id").sort_index()
    # Compare the selected series details with the reference and stop if they differ.
    pd.testing.assert_frame_equal(sheet_metadata[definition_fields], baseline_metadata[definition_fields], obj=f"{dataset}/{sheet}: series definitions")

    # Reshape the monthly values into date, series ID and value columns.
    sheet_observations = (
        raw_sheet.iloc[first_data_row:].set_axis(["date", *series_ids], axis=1).melt("date", var_name="series_id").astype({"date": "datetime64[ns]"})
    )
    # Confirm that this sheet reaches the requested release month.
    assert sheet_observations["date"].max() == download_month, f"{dataset}/{sheet}: latest month does not match {release_month}"
    # Convert the observation values to numbers.
    sheet_observations["value"] = pd.to_numeric(sheet_observations["value"])
    # Collect this sheet's values and series details, adding their source labels.
    observation_frames.append(sheet_observations.assign(dataset=dataset))
    metadata_frames.append(sheet_metadata.reset_index().assign(dataset=dataset, sheet=sheet))
    # Record the number of months, series and observations, plus the latest month.
    sheet_checks.append((dataset, sheet, sheet_observations["date"].nunique(), len(sheet_metadata), len(sheet_observations), sheet_observations["date"].max()))

# Combine the results from all data sheets into two tables.
table_010_metadata = pd.concat(metadata_frames, ignore_index=True)
table_010 = pd.concat(observation_frames, ignore_index=True)
# Check that no series ID appears in more than one sheet.
assert not table_010_metadata.duplicated(["dataset", "series_id"]).any(), f"{dataset}: a series ID is repeated across sheets"
# Display a summary of the records read from each data sheet.
display(pd.DataFrame(sheet_checks, columns=["dataset", "sheet", "months", "series", "observations", "latest_month"]))

## 5. Tidy the output columns

In [ ]:
# Remove surrounding spaces and trailing semicolons, then split each description into columns.
description_parts = table_010_metadata["description"].str.strip().str.rstrip(";").str.split(";", expand=True)
# Remove spaces and leading > markers from each part of the description.
description_parts = description_parts.apply(lambda s: s.str.strip().str.lstrip(">").str.strip())
# Copy the first three description parts into the measure, sex and region columns.
table_010_metadata["measure"] = description_parts[0]
table_010_metadata["sex"] = description_parts[1]
table_010_metadata["region"] = description_parts[2]
# Label every series as covering people aged 15 years and over.
table_010_metadata["age"] = "15 years and over"
# Add an apostrophe to the thousands unit label; leave other unit labels unchanged.
table_010_metadata["unit"] = table_010_metadata["unit"].replace("000", "'000")
# Check that every series has a measure, sex, region and age label.
assert table_010_metadata[["measure", "sex", "region", "age"]].notna().all().all()

# Count the rows for each dataset, date and series ID combination before joining.
observation_keys = ["dataset", "date", "series_id"]
keys_before_join = table_010[observation_keys].value_counts(sort=False).sort_index()

# Check that each dataset and series ID has only one matching row of series details.
table_010 = table_010.merge(
    table_010_metadata[["dataset", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit"]],
    on=["dataset", "series_id"],
    validate="many_to_one",
)
# Check that every dataset, date and series ID combination still has the same row count.
pd.testing.assert_series_equal(
    keys_before_join, table_010[observation_keys].value_counts(sort=False).sort_index(), obj="Table 010 metadata join: observation keys and counts"
)
# Display the table size and the first five rows for review.
print(table_010.shape)
display(table_010[["date", "series_id", "measure", "region", "sex", "age", "series_type", "data_type", "unit", "value"]].head())

## 6. Save the cleaned CSV

Save the cleaned data in this job run's output folder. Rerunning the same source within the same job run overwrites its raw workbook and CSV.

In [ ]:
# Select the columns and their order for the cleaned output.
df = table_010[["date", "series_id", "measure", "sex", "age", "region", "series_type", "data_type", "unit", "value"]].copy()

# Check that the prepared data reaches the requested release month.
assert df["date"].notna().all(), f"{dataset}: missing observation dates"
assert df["date"].max() == download_month, f"{dataset}: latest month does not match {release_month}"

# Save the cleaned table with the same columns as the original CSV output.
csv_file = run / f"{dataset_key}_{release_month}.csv"
df.to_csv(csv_file, index=False, date_format="%Y-%m-%d")
print(df.shape)
display(df.head())

## 7. Publish the Delta table

Replace `YOUR_CATALOG` and `YOUR_SCHEMA` with your existing destination names. `dataset_key` supplies this source's table name. Run this cell in Databricks after the CSV export.

The write creates a missing Delta table or overwrites its data. It uses the cleaned `df` already in memory; no staging Delta table or separate publisher is needed. This source updates independently of the other two. Plain overwrite does not automatically replace an existing table's schema.

In [ ]:
# Enter the destination catalogue and schema.
catalog = "YOUR_CATALOG"
schema = "YOUR_SCHEMA"
table = f"{catalog}.{schema}.{dataset_key}"

# Convert the cleaned data to Spark and keep dates without a time component.
sdf = spark.createDataFrame(df)
sdf = sdf.withColumn("date", sdf["date"].cast("date"))

# Create the Delta table or overwrite its existing data.
sdf.write.format("delta").mode("overwrite").saveAsTable(table)

print(f"Published {len(df):,} rows to {table}")